In [28]:
from pygmtsar import S1, Stack, tqdm_dask, ASF, Tiles, XYZTiles
import json
import geopandas as gpd
import pandas as pd
import datetime
import warnings

In [2]:
buoy_data = pd.read_csv('IABP_LEVEL1_2020.csv')


In [3]:
buoy_data = buoy_data[buoy_data['BuoyID']==300234066312080]
#buoy_data

In [4]:
entry_index = 0


In [6]:
data_dir = 'pgmt_data_dir'
work_dir = 'pgmt_work_dir'


In [37]:
match = {}

d1 = datetime.datetime(1999, 1, 1).strftime('%Y-%m-%d')

for entry_index in range(len(buoy_data)):

    entry = buoy_data.iloc[entry_index]
    date = datetime.datetime(int(entry['Year']), int(entry['Month']), int(entry['Day']))
    if (date < datetime.datetime.strptime(d1, '%Y-%m-%d')):
        continue
    
    d0 = date.strftime('%Y-%m-%d')# alt : %Y-%m-%d %H:%M:%S
    d1 = (date + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
    lat, lon = entry['Lat'], entry['Lon']

    if (lon) > 180: lon -= 360
    
    geojson = f'''{{
        "type": "Feature",
        "geometry": {{
            "type": "Polygon",
            "coordinates": [[
                [{lon-0.01},{lat-0.01}],
                [{lon-0.01},{lat+0.01}],
                [{lon+0.01},{lat+0.01}],
                [{lon+0.01},{lat-0.01}]
            ]]
        }},
        "properties": {{}}
    }}
    '''

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        AOI = gpd.GeoDataFrame.from_features([json.loads(geojson)])
        try:
            bursts = ASF.search(geometry=AOI, startTime=d0, stopTime=d1)
        except ValueError as e:
            if "Assigning CRS to a GeoDataFrame without a geometry column" in str(e):
                bursts = gpd.GeoDataFrame(columns = ['fileID'])
            else:
                raise
    
    BURSTS = bursts.fileID.tolist()

    if len(BURSTS) > 0:
        match[entry_index] = BURSTS
        
#print(asf.download(data_dir, SCENES, 

/home/adminix/sgconda/lib/python3.12/site-packages/pygmtsar/ASF.py:675: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'LineString' and geometry.coords[0] == geometry.coords[-1]:
/home/adminix/sgconda/lib/python3.12/site-packages/pygmtsar/ASF.py:677: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'Polygon':
["'type': 'WRAP': 'report': 'Wrapped 5 value(s) to +/-180 longitude'"]
/home/adminix/sgconda/lib/python3.12/site-packages/pygmtsar/ASF.py:675: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'LineString' and geometry.coords[0] == geometry.coords[-1]:
/home/adminix/sgconda/lib/python3.12/site-packages/pygmtsar/ASF.py:677: ShapelyDe

In [38]:
match

{23: ['S1_236972_IW1_20200103T073421_VV_AA07-BURST'],
 46: ['S1_236972_IW1_20200103T073421_VV_AA07-BURST'],
 69: ['S1_313970_IW1_20200105T183402_VV_CB23-BURST',
  'S1_313969_IW1_20200105T183400_VV_CB23-BURST'],
 93: ['S1_313970_IW1_20200105T183402_VV_CB23-BURST'],
 184: ['S1_094880_IW3_20200110T184214_VV_507B-BURST'],
 207: ['S1_313971_IW2_20200111T183324_VV_CB4C-BURST',
  'S1_313970_IW2_20200111T183321_VV_CB4C-BURST'],
 228: ['S1_313972_IW2_20200111T183327_VV_CB4C-BURST',
  'S1_313971_IW2_20200111T183324_VV_CB4C-BURST'],
 344: ['S1_313972_IW2_20200117T183408_VV_179B-BURST',
  'S1_313971_IW2_20200117T183405_VV_179B-BURST'],
 368: ['S1_313972_IW2_20200117T183408_VV_179B-BURST',
  'S1_313971_IW2_20200117T183405_VV_179B-BURST'],
 484: ['S1_313972_IW3_20200123T183327_VV_E73F-BURST',
  'S1_313971_IW3_20200123T183324_VV_E73F-BURST'],
 507: ['S1_313971_IW3_20200123T183324_VV_E73F-BURST',
  'S1_313970_IW3_20200123T183322_VV_E73F-BURST'],
 625: ['S1_313971_IW3_20200129T183406_VV_8E47-BURST'],
 

In [39]:
BURSTS = match[23]

In [44]:
asf_user = 'mjeiouws'
asf = ASF(asf_user, 'MORYCORis343!')
print(asf.download(data_dir, BURSTS))

ASF Downloading Bursts Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ASF Downloading Sentinel-1 SLC Bursts:   0%|          | 0/1 [00:00<?, ?it/s]

Warning 1: PROJ: internal_proj_create_from_database: /home/adminix/sgconda/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 5 whereas a number >= 6 is expected. It comes from another PROJ installation.
Warning 1: PROJ: internal_proj_create_from_name: /home/adminix/sgconda/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 5 whereas a number >= 6 is expected. It comes from another PROJ installation.


                                burst_or_scene
0  S1_236972_IW1_20200103T073421_VV_AA07-BURST


In [47]:
from osgeo import gdal

In [48]:
BURSTS[0]

'S1_236972_IW1_20200103T073421_VV_AA07-BURST'

In [74]:
#burst_img = gdal.Open('pgmt_data_dir/S1A_IW_SLC__1SDV_20200103T073421_20200103T073424_030633_038297_AA07.SAFE/measurement/s1a-iw1-slc-vv-20200103t073421-20200103t073424-030633-038297-001.tiff')
gdal.UseExceptions()
scene_name = 'pgmt_data_dir/S1A_IW_SLC__1SDV_20200103T073421_20200103T073424_030633_038297_AA07.SAFE'
burst_img = gdal.OpenEx(scene_name, allowed_drivers=['SAFE'])



RuntimeError: `pgmt_data_dir/S1A_IW_SLC__1SDV_20200103T073421_20200103T073424_030633_038297_AA07.SAFE' not recognized as being in a supported file format.

In [72]:
burst_img

In [65]:
print(dir(burst_img))

burst_img.GetDriver().Name()

['__bool__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__']


AttributeError: 'NoneType' object has no attribute 'GetDriver'

In [61]:
burst_img.GetDriver().GetName()

'GTiff'